# Scan2Stage M3/M4
UGScan ZIP/GLB → room normalization → plane stripping → object candidates.


In [ ]:
REPO='https://github.com/6564200/Scan2Stage.git'
WORKDIR='/content/Scan2Stage'
!rm -rf {WORKDIR}
!git clone --branch main --single-branch {REPO} {WORKDIR}
%cd {WORKDIR}
!bash scripts/colab_bootstrap.sh
from google.colab import files
from pathlib import Path
uploaded=files.upload()
name=next(iter(uploaded))
INPUT=Path('/content/Scan2Stage/data')/Path(name).name
INPUT.parent.mkdir(parents=True,exist_ok=True)
Path(name).replace(INPUT)


In [ ]:
import subprocess,json
OUT=Path('/content/Scan2Stage/outputs/m4')
OUT.mkdir(parents=True,exist_ok=True)
p=subprocess.run(['scan2stage',str(INPUT),'--output-dir',str(OUT),'--samples','300000','--source-up','y'],text=True,capture_output=True)
print(p.stdout); print(p.stderr)
assert p.returncode==0, f'scan2stage failed: {p.returncode}'
room=json.loads((OUT/'room_geometry.json').read_text())
obj=json.loads((OUT/'object_candidates.json').read_text())
print('Rectangle:',room['rectangle'])
print('Outside-room:',obj['outside_room_points'])
print('Before voxel:',obj['object_points_before_voxel'])
print('After voxel:',obj['object_points_after_voxel'])
print('Removed large planes:',obj['removed_large_plane_count'])
print('After plane strip:',obj['object_points'])
print('DBSCAN noise:',obj['dbscan_noise_points'])
print('Candidates:',obj['candidate_count'])
for x in obj['removed_large_planes']: print('plane',x['inliers'],[round(v,3) for v in x['extents_m']])
for c in obj['candidates']: print(c['id'],[round(v,3) for v in c['center_m']],[round(v,3) for v in c['extents_m']],c['point_count'])
